# 💻 Feilsøking: å lese feilmeldinger

## Oversikt

Så langt har forelesningsnotebookene vist deg kode som *virker*. Virkeligheten er
at en stor del av all programmering går med til å finne ut hvorfor koden *ikke*
virker – og til å lese feilmeldinger. En feilmelding er ikke en irettesettelse;
det er den mest presise informasjonen du kommer til å få om hva som er galt.

Denne notebooken utløser feil med vilje, slik at du kjenner dem igjen når de
dukker opp i dine egne øvinger og i prosjektet. Vi ser på:

- **Anatomien i en traceback** – hvordan lese feilmeldingen nedenfra og opp
- **De vanligste Python-feilene** – `NameError`, `TypeError`, `IndexError`, `KeyError` ...
- **Feil du møter med pandas og geopandas** – filstier, kolonnenavn, feil datatype
- **Når det ikke kommer noen feilmelding** – advarsler og «stille» feil som gir feil svar
- **En feilsøkingsarbeidsflyt** du kan følge hver gang

```{note}
Flere av kodecellene her er *ment* å feile. Feilmeldingen er en del av pensum – les den, ikke hopp
over den.
```


## 1. Anatomien i en traceback

Når Python møter en feil den ikke kan håndtere, stopper den og skriver ut en
**traceback** («tilbakesporing»). Den kan se skummel ut, men har alltid samme
struktur:

- **Nederste linje**: feiltypen (f.eks. `ZeroDivisionError`) og en melding.
  **Les denne først.**
- **Over det**: hvilken linje i koden som feilet, med en pil (`---->`).
- **Enda lenger opp**: hele «kjeden» av funksjonskall som ledet dit – det nyeste
  kallet nederst.

Kjør cellen under. Den regner ut et gjennomsnitt, men får en tom liste.


In [ ]:
def gjennomsnitt(tall):
    return sum(tall) / len(tall)


def lag_rapport(data):
    return f"Gjennomsnittet er {gjennomsnitt(data)}"


lag_rapport([])


Les tracebacken nedenfra:

1. Siste linje: `ZeroDivisionError: division by zero`. Vi delte på null.
2. Pilen peker på `return sum(tall) / len(tall)` inne i `gjennomsnitt` – det er
   der delingen skjer.
3. Rammen over viser at `gjennomsnitt` ble kalt fra `lag_rapport`, som ble kalt
   fra `lag_rapport([])`.

**Den egentlige feilen er ikke i `gjennomsnitt`** – den er at noen sendte inn en
tom liste. Linjenummeret Python gir deg er der feilen *oppstod*, ikke alltid der
*årsaken* ligger. En `assert len(tall) > 0, "listen er tom"` øverst i funksjonen
(som i uke 2) ville gjort feilmeldingen tydeligere.


## 2. De vanligste Python-feilene

### `NameError` – et navn Python ikke kjenner


In [ ]:
koordinater = [10.75, 59.91]

print(kordinater)   # skrivefeil: mangler en «o»


`NameError: name 'kordinater' is not defined`. Nesten alltid én av to ting:

- en **skrivefeil** i variabel- eller funksjonsnavnet, eller
- du har **ikke kjørt cellen** der navnet ble definert (eller kjørte cellene i
  feil rekkefølge).

I en notebook: sjekk at du har kjørt cellene over, ovenfra og ned.
`Kernel -> Restart & Run All` fjerner all tvil om rekkefølge.

### `TypeError` – en operasjon på feil datatype


In [ ]:
antall_punkt = "120"

print(antall_punkt + 5)


`TypeError: can only concatenate str (not "int") to str`. `antall_punkt` er en
streng (`"120"` med hermetegn), ikke et tall. Feilmeldingen sier nøyaktig hvilke
to typer som ikke passer sammen. Fiks: `int(antall_punkt) + 5`.

Dette er grunnen til at `02_assertions` lot `TypeError` passere i stedet for å
fange den med en `assert` – meldingen er allerede tydelig nok.

### `IndexError` og `KeyError` – noe finnes ikke


In [ ]:
byer = ["Oslo", "Bergen", "Trondheim"]

print(byer[5])


`IndexError: list index out of range`. Lista har tre element (indeks 0, 1, 2), og
det finnes ingen indeks 5. Husk at Python teller fra 0, så siste element er
`byer[len(byer) - 1]` eller `byer[-1]`.

`KeyError` er den samme feilen for ordbøker:


In [ ]:
attributter = {"navn": "Ås", "befolkning": 10962}

print(attributter["areal"])


`KeyError: 'areal'` – det finnes ingen nøkkel `"areal"`. Skriv
`attributter.keys()` for å se hvilke nøkler som faktisk finnes.

### `SyntaxError` og `IndentationError` – Python skjønner ikke koden

Disse to oppstår *før* koden kjører, så de kan ikke «fanges». De vanligste er
manglende kolon og feil innrykk:

```python
for by in byer
    print(by)
```

gir

```
  Cell In[1], line 1
    for by in byer
                 ^
SyntaxError: expected ':'
```

Python viser deg omtrent hvor den ble forvirret med en `^`. Se etter manglende
`:`, ubalanserte parenteser `()` / klammer `[]`, eller et innrykk som ikke stemmer
med linjene rundt.


## 3. Feil du møter med pandas og geopandas

Her lager vi et lite datasett å feile på, uten å måtte lese noen fil:


In [ ]:
import geopandas as gpd
from shapely.geometry import Point

byer = gpd.GeoDataFrame(
    {"navn": ["Oslo", "Bergen", "Ås"], "befolkning": [709037, 291940, 10962]},
    geometry=[Point(10.75, 59.91), Point(5.32, 60.39), Point(10.78, 59.66)],
    crs="EPSG:4326",
)
byer


### `FileNotFoundError` / `DriverError` – stien er feil


In [ ]:
data = gpd.read_file("data/finnes_ikke.gpkg")


geopandas klarer ikke å åpne en fil som ikke er der. Sjekk:

- Er stien **relativ til der notebooken kjører**? `pathlib.Path().resolve()` viser
  arbeidsmappa (se `03_filstier`).
- Finnes fila? `pathlib.Path("data/finnes_ikke.gpkg").exists()` gir `True`/`False`.
- Er filnavn og filendelse riktig skrevet?


### `KeyError` – kolonnenavnet stemmer ikke


In [ ]:
byer["Befolkning"]


`KeyError: 'Befolkning'`. Kolonnen heter `befolkning` med liten b. Kolonnenavn er
følsomme for store/små bokstaver og skjulte mellomrom. Sjekk alltid hva de heter:


In [ ]:
print(list(byer.columns))


Hvis et navn har et mellomrom du ikke ser (`"befolkning "`), kan du rydde opp med
`byer.columns = byer.columns.str.strip()`.

### `AttributeError` – metoden eller egenskapen finnes ikke


In [ ]:
byer.to_crs("EPSG:25833").centorid


`AttributeError: 'GeoDataFrame' object has no attribute 'centorid'`. Skrivefeil
for `centroid`. Samme feil får du hvis du kaller en geometri-metode på et vanlig
`DataFrame` (uten geometrikolonne), eller bruker et gammelt metodenavn som er
fjernet i en nyere versjon av pakka.


### Et sjekkpunkt for uke 5

En feil som *ikke* gir noen feilmelding: hvis du gjør en romlig kobling
(`sjoin`, uke 5) mellom to lag med **forskjellig CRS**, får du ofte et helt tomt
resultat i stedet for en feil. Derfor sjekker forelesningsnotebookene alltid
`assert lag_a.crs == lag_b.crs` før en kobling. Mer om dette i uke 5.


## 4. Når det ikke kommer noen feilmelding

De farligste feilene er de som gir deg et svar – bare feil svar.

### Advarsel: geometrisk beregning i et geografisk CRS


In [ ]:
from shapely.geometry import Polygon

ruter = gpd.GeoDataFrame(
    {"id": [1, 2]},
    geometry=[
        Polygon([(10.7, 59.9), (10.8, 59.9), (10.8, 60.0), (10.7, 60.0)]),
        Polygon([(10.8, 59.9), (10.9, 59.9), (10.9, 60.0), (10.8, 60.0)]),
    ],
    crs="EPSG:4326",
)

ruter.area


Koden kjører, men skriver ut en `UserWarning: Geometry is in a geographic CRS.
Results from 'area' are likely incorrect.` og noen bittesmå tall. Arealet regnes
ut i **grader**, ikke meter. En advarsel (`Warning`) stopper ikke koden – men den
skal tas like alvorlig som en feil. Fiks: reprojiser først.


In [ ]:
ruter.to_crs("EPSG:25833").area


Nå får du areal i kvadratmeter, som forventet.

### `SettingWithCopyWarning` – du endrer kanskje ikke det du tror


In [ ]:
store = byer[byer["befolkning"] > 100000]
store["kategori"] = "storby"


`store` er her bare et *utsnitt* (view) av `byer`, ikke en selvstendig kopi. På
eldre pandas-versjoner gir dette en `SettingWithCopyWarning` om at tilordningen
kanskje ikke gjør det du forventer; på nyere versjoner kan endringen i stillhet
ikke slå gjennom på `byer`. Uansett: når du skal *endre* et filtrert datasett, be
eksplisitt om en kopi.


In [ ]:
store = byer[byer["befolkning"] > 100000].copy()
store["kategori"] = "storby"
store


### Stille feil: et tomt resultat


In [ ]:
byer[byer["navn"] == "oslo"]


Ingen feil – men null rader, fordi `"oslo"` ikke er det samme som `"Oslo"`. Når
et filter gir uventet lite, sjekk:

- `len(resultat)` eller `resultat.shape`
- de faktiske verdiene: `byer["navn"].unique()`
- store/små bokstaver og mellomrom

### Ugyldig geometri


In [ ]:
sloyfe = Polygon([(0, 0), (1, 1), (1, 0), (0, 1)])   # kantene krysser hverandre
print(sloyfe.is_valid)


`False`. Polygonet krysser seg selv. Slike geometrier gir ofte ikke feil med det
samme, men kan gi feil areal, eller en `TopologyException` senere i en
overlay-operasjon (uke 7). De fleste kan repareres med `make_valid`:


In [ ]:
import shapely

reparert = shapely.make_valid(sloyfe)
print(reparert.is_valid)


## 5. En feilsøkingsarbeidsflyt

Når noe feiler, gjør dette i rekkefølge:

1. **Les den siste linja i tracebacken.** Feiltype + melding sier som regel hva
   som er galt.
2. **Finn linjenummeret** (pilen `---->`), men se også på linjene rett over –
   årsaken kan ligge tidligere.
3. **Sjekk verdiene.** Skriv ut det du tror du har:
   `print(x)`, `type(x)`, `x.shape`, `x.dtypes`, `x.crs`, `list(x.columns)`,
   `x.head()`.
4. **Isoler.** Kjør en liten del av cella for seg, eller del cella i to.
5. **Lag et minimalt eksempel** med bare noen få rader – ofte ser du feilen da.
6. **Bruk `assert`** til å sjekke antakelsene dine tidlig (CRS, at en kolonne
   finnes, at et datasett ikke er tomt).
7. **Søk på feilmeldingsteksten.** Bytt ut dine egne variabelnavn med generelle
   ord. Du kan også be et KI-verktøy forklare meldingen – men behandle svaret som
   et forslag du må verifisere, ikke en fasit (se siden om bruk av KI).


## 6. Sjekk din forståelse

### Oppgave 1

Cellen under feiler. Uten å kjøre den: hvilken feiltype får du, og hva er
årsaken?

```python
kommuner = gpd.GeoDataFrame(
    {"navn": ["Ås", "Vestby"], "innbyggere": [23000, 18000]},
    geometry=[Point(10.78, 59.66), Point(10.75, 59.60)],
    crs="EPSG:4326",
)

kommuner["befolkning"].sum()
```

```{admonition} Løsningsforslag
:class: dropdown

`KeyError: 'befolkning'`. Kolonnen heter `innbyggere`, ikke `befolkning`. Sjekk
med `list(kommuner.columns)`. Feilen kommer fra `kommuner["befolkning"]`, før
`.sum()` i det hele tatt kjører.
```

### Oppgave 2

Denne koden kjører uten feil, men resultatet er feil. Hva er galt?

```python
kommuner.to_crs("EPSG:4326")
kommuner.geometry.area
```

```{admonition} Løsningsforslag
:class: dropdown

To ting:

1. `kommuner.to_crs("EPSG:4326")` returnerer en *ny* GeoDataFrame – den endrer
   ikke `kommuner`. Uten `kommuner = kommuner.to_crs(...)` skjer ingenting.
2. Selv med tilordning er 4326 et geografisk CRS, så `.area` gir en `UserWarning`
   og tall i grader. Reprojiser til et projisert CRS (f.eks. `EPSG:25833`) før du
   regner areal.
```


## 7. Oppsummering

| Feil / advarsel | Vanlig årsak | Hva du sjekker |
|---|---|---|
| `NameError` | Skrivefeil, eller celle ikke kjørt | Stavemåte; kjør cellene ovenfra og ned |
| `TypeError` | Operasjon på feil datatype | `type(x)`; er «tallet» egentlig en streng? |
| `IndexError` / `KeyError` | Element / nøkkel / kolonne finnes ikke | `len()`, `.keys()`, `list(df.columns)` |
| `SyntaxError` / `IndentationError` | Manglende `:`, ubalansert `()`/`[]`, feil innrykk | `^`-markøren; linjene rundt |
| `FileNotFoundError` / `DriverError` | Feil sti eller filnavn | `Path(...).exists()`, `Path().resolve()` |
| `AttributeError` | Skrivefeil i metodenavn, feil objekttype, utdatert API | Stavemåte; `type(x)`; pakkeversjon |
| `UserWarning` om geografisk CRS | `.area` / `.length` / `.centroid` i EPSG:4326 | `df.crs`; reprojiser med `to_crs()` |
| `SettingWithCopyWarning` | Endrer et utsnitt, ikke en kopi | Legg til `.copy()` etter filtreringen |
| Tomt resultat, ingen feil | Store/små bokstaver, mellomrom, CRS-mismatch | `.shape`, `.unique()`, `assert a.crs == b.crs` |

Feilmeldinger er en ferdighet du bygger opp ved å møte dem. Hver gang du løser
én, går neste gang raskere.
